In [1]:
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [2]:
# Load Dataset (SST‑2)
dataset = load_dataset("glue", "sst2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [14]:
# Tokenizer + Model

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",  #DistilBERT
    num_labels=2
)


# Tokenization

def tokenize_function(example):
    return tokenizer(
        example["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)


# Rename column on each split separately

tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Set format for PyTorch
tokenized_datasets.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [17]:
# Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)


# Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)


# Train

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.135588,0.291295,0.910550,0.911565
2,0.175379,0.355087,0.907110,0.910299


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8420, training_loss=0.17384995040339982, metrics={'train_runtime': 1538.6693, 'train_samples_per_second': 87.542, 'train_steps_per_second': 5.472, 'total_flos': 4460773416041472.0, 'train_loss': 0.17384995040339982, 'epoch': 2.0})

In [18]:
# Evaluate

results = trainer.evaluate()
print("\n Evaluation Results:", results)


# Confusion Matrix

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="binary")
    }

predictions = trainer.predict(tokenized_datasets["validation"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids
cm = confusion_matrix(labels, preds)
print("\nConfusion Matrix:\n", cm)




 Evaluation Results: {'eval_loss': 0.3550874888896942, 'eval_accuracy': 0.9071100917431193, 'eval_f1': 0.9102990033222591, 'eval_runtime': 3.3853, 'eval_samples_per_second': 257.585, 'eval_steps_per_second': 16.247, 'epoch': 2.0}

Confusion Matrix:
 [[380  48]
 [ 33 411]]


In [19]:
# # Custom Prediction

# def predict_sentiment(text):
#     inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
#     with torch.no_grad():
#         outputs = model(**inputs)
#     probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
#     return {"negative": float(probs[0][0]), "positive": float(probs[0][1])}



In [21]:
def predict_sentiment(text):
    # Tokenize and move tensors to the same device as the model
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)

    # Move all input tensors to the model's device
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    # Move probabilities back to CPU for conversion to float
    return {"negative": float(probs[0][0].cpu()), "positive": float(probs[0][1].cpu())}

In [ ]:
con=1

In [31]:
while True:
    text = input("Enter a sentence: ")
    result = predict_sentiment(text)   # store the returned dict
    print(f"Sentiment: {result}")      # display it

    answer = input("Do you want to predict another sentence? (yes/no): ").lower()
    if answer != 'yes':
        break

Enter a sentence: wow
Sentiment: {'negative': 0.0014332450227811933, 'positive': 0.9985668063163757}
Do you want to predict another sentence? (yes/no): yes
Enter a sentence: shit
Sentiment: {'negative': 0.9964728951454163, 'positive': 0.0035270885564386845}
Do you want to predict another sentence? (yes/no): yes
Enter a sentence: Daaaaaaamn
Sentiment: {'negative': 0.9766428470611572, 'positive': 0.023357169702649117}
Do you want to predict another sentence? (yes/no): yes
Enter a sentence: haha
Sentiment: {'negative': 0.614084780216217, 'positive': 0.38591521978378296}
Do you want to predict another sentence? (yes/no): no
